# Chapitre 8 - Lab : verrou Redis, pause de processus et jetons de clôture

Le cours 4 a simulé le scénario le plus célèbre du chapitre : un client obtient un bail,
se fige, son bail expire, un autre client le prend, et le premier se réveille pour écrire
comme si de rien n'était. Ce lab le reproduit avec de **vrais** composants : un verrou à
bail dans Redis (`SET ... NX PX`), un stockage (un second Redis) et deux clients qui
tournent dans des conteneurs. Pour figer un client, on utilise `docker pause`, qui gèle
tous ses processus (comme une longue pause du ramasse-miettes ou une VM suspendue).
Puis on corrige avec des **jetons de clôture** vérifiés par le stockage. Sections du livre :
*Process Pauses*, *The leader and the lock*, *Fencing tokens*.

**Plan.**

0. Démarrer les services et vérifier la connexion.
1. Un bail dans Redis : `SET NX PX`.
2. Le zombie : `docker pause` pendant que le bail expire.
3. La parade : des jetons de clôture vérifiés par le stockage.
4. Ce que le processus figé a vu.
5. Pour aller plus loin, et arrêt des services.

## 0. Démarrer les services et vérifier la connexion

Le fichier [`docker/distsys/compose.yml`](../../docker/distsys/compose.yml) définit quatre
conteneurs : `lock` (Redis qui sert de service de verrous, port hôte 10901), `storage`
(Redis qui joue le service de stockage, port 10902), et `worker1`, `worker2`, deux
clients qui exécutent le script [`docker/distsys/worker.sh`](../../docker/distsys/worker.sh)
avec `redis-cli`. Depuis la racine du dépôt :

```bash
docker compose -f docker/distsys/compose.yml -p ddia_distsys up -d     # démarrer
docker compose -f docker/distsys/compose.yml -p ddia_distsys down -v   # arrêter et nettoyer
```

Le script d'un client fait, dans l'ordre : obtenir le bail ; « préparer » l'écriture
(`TRAVAIL` secondes) ; **vérifier** que le bail lui appartient toujours ; travailler encore
une seconde ; **écrire** dans le stockage. C'est la boucle naïve du livre : vérifier, puis
agir. Chaque étape est journalisée dans une liste `evenements` du stockage, avec un
horodatage lu dans `/proc/uptime`, une horloge **monotone** commune à tous les
conteneurs puisqu'ils partagent le noyau de la machine hôte.

In [1]:
import subprocess
import time

import matplotlib.pyplot as plt
import pandas as pd
import redis

from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16

PROJET = "ddia_distsys"
W1, W2 = f"{PROJET}-worker1-1", f"{PROJET}-worker2-1"
verrous = redis.Redis(host="localhost", port=10901, decode_responses=True)
stockage = redis.Redis(host="localhost", port=10902, decode_responses=True)
try:
    print("service de verrous :", verrous.ping(), "; stockage :", stockage.ping(), "; Redis", verrous.info()["redis_version"])
    etat = subprocess.run(["docker", "inspect", "-f", "{{.State.Status}}", W1, W2], capture_output=True, text=True, check=True)
    print("clients :", etat.stdout.split())
except Exception as exc:                                   # noqa: BLE001
    raise SystemExit("Services absents : lancez d'abord docker compose ... up -d (voir ci-dessus).") from exc


def maintenant_ms():
    """Horloge monotone de l'hôte, la même que celle des conteneurs (/proc/uptime)."""
    with open("/proc/uptime") as f:
        return int(float(f.read().split()[0]) * 1000)


def docker(*args):
    subprocess.run(["docker", *args], check=True, capture_output=True)

service de verrous : True ; stockage : True ; Redis 7.4.11
clients : ['running', 'running']
